# 위험 소리 분류 파인튜닝 Colab

이 노트북은 회의록/중간보고서에 적힌 후보 모델과 데이터셋을 기준으로, 청각장애인 지원 키링/앱 서비스에서 중요한 **위험·주의 소리**를 우선 분류하도록 구성했습니다.

- 기본 타깃 클래스: `car_horn`, `siren`, `glass_breaking`, `explosion_or_gunshot`, `construction_or_machine`, `fire_alarm`, `baby_cry`, `doorbell_or_bell`, `other`
- 기본 자동 다운로드: ESC-50
- 선택 다운로드/로딩: UrbanSound8K, FSD50K, AudioSet, AI Hub 소음 환경 음성인식 데이터
- 위험 소리 성능 보강: 클래스 가중치, `other` 샘플 제한, augmentation, validation F1 확인

> Colab에서 먼저 `런타임 > 런타임 유형 변경 > GPU`를 선택하세요. 빠른 실험은 `EPOCHS=3`, `MAX_SAMPLES_PER_CLASS=300` 정도로 시작하고, 최종 비교 때 늘리는 방식이 안전합니다.

## 모델: AST / Audio Spectrogram Transformer
Hugging Face `ASTForAudioClassification`를 프로젝트 라벨 수에 맞춰 classification head를 교체한 뒤 fine-tuning합니다.

In [5]:
# Colab/VS Code 설치 셀: AST / Hugging Face Transformers
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers",
        "datasets",
        "evaluate",
        "accelerate",
        "librosa",
        "soundfile",
        "pandas",
        "scikit-learn",
        "tqdm",
        "kaggle",
        "matplotlib",
    ],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'transformers', 'datasets', 'evaluate', 'accelerate', 'librosa', 'soundfile', 'pandas', 'scikit-learn', 'tqdm', 'kaggle', 'matplotlib'], returncode=0)

In [6]:
import os
import re
import json
import math
import random
import zipfile
import shutil
import subprocess
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# =========================================================
# 1) 데이터셋 설정
# =========================================================
DATA_ROOT = Path('/content/sound_data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# 회의록 사진/보고서에 있는 데이터셋을 모두 지원합니다.
# ESC-50은 자동 다운로드, UrbanSound8K는 Kaggle 계정이 있으면 자동 다운로드,
# FSD50K/AudioSet/AI Hub는 크기·라이선스·로그인 이슈 때문에 Google Drive에 받아둔 경로를 연결하는 방식입니다.
DATASETS_TO_USE = [
    'esc50',
    'urbansound8k',
    'fsd50k',
    'audioset',
    'aihub_noise',
]

USE_KAGGLE_URBANSOUND8K = False  # Kaggle API 토큰(kaggle.json)을 넣었다면 True로 바꾸세요.

# 큰 데이터셋은 Drive에 직접 다운로드한 뒤 아래 경로를 맞추세요.
FSD50K_ROOT = Path('/content/drive/MyDrive/datasets/FSD50K')
AUDIOSET_ROOT = Path('/content/drive/MyDrive/datasets/AudioSet_prepared')
AIHUB_NOISE_ROOT = Path('/content/drive/MyDrive/datasets/AIHub_noise_speech')

# Colab 빠른 실험용 제한. 최종 실험 때 늘리거나 None으로 바꾸세요.
MAX_SAMPLES_PER_CLASS = 600
MAX_OTHER_SAMPLES = 600
MIN_SAMPLES_PER_CLASS = 5

# 위험/주의 분류 타깃. 실제 서비스에서는 필요에 따라 baby_cry/doorbell을 별도 "주의" 레벨로 매핑하면 됩니다.
TARGET_LABELS = [
    'car_horn',
    'siren',
    'glass_breaking',
    'explosion_or_gunshot',
    'construction_or_machine',
    'fire_alarm',
    'baby_cry',
    'doorbell_or_bell',
    'other',
]
DANGER_LABELS = {
    'car_horn', 'siren', 'glass_breaking', 'explosion_or_gunshot',
    'construction_or_machine', 'fire_alarm'
}

# 라벨 매핑 우선순위: 여러 라벨이 동시에 있을 때 위험도가 높은 쪽을 선택합니다.
LABEL_PRIORITY = [
    'explosion_or_gunshot',
    'glass_breaking',
    'fire_alarm',
    'siren',
    'car_horn',
    'construction_or_machine',
    'baby_cry',
    'doorbell_or_bell',
    'other',
]

KEYWORD_RULES = [
    ('explosion_or_gunshot', [
        'explosion', 'explode', 'blast', 'gun_shot', 'gunshot', 'gun shot', 'gunfire', 'fireworks', 'firework',
        '폭발', '폭발음', '총성', '총소리', '사격', '폭죽'
    ]),
    ('glass_breaking', [
        'glass_breaking', 'glass breaking', 'glass_break', 'breaking_glass', 'shatter', 'glass',
        '유리', '유리깨짐', '유리 깨', '파손'
    ]),
    ('fire_alarm', [
        'fire_alarm', 'smoke_alarm', 'smoke detector', 'alarm', 'beep', 'buzzer',
        '화재경보', '화재 경보', '경보기', '경보음', '경보'
    ]),
    ('siren', [
        'siren', 'civil_defense_siren', 'emergency_vehicle', 'ambulance', 'police_siren',
        '싸이렌', '사이렌', '구급차', '경찰차'
    ]),
    ('car_horn', [
        'car_horn', 'vehicle_horn', 'air_horn', 'horn', 'honk',
        '경적', '경적소음', '클락션', '자동차 경적'
    ]),
    ('construction_or_machine', [
        'jackhammer', 'drilling', 'drill', 'chainsaw', 'hand_saw', 'saw', 'hammer', 'construction', 'machine',
        '착암', '드릴', '전동해머', '해머드릴', '기계톱', '절단기', '공사', '공장', '프레스'
    ]),
    ('baby_cry', [
        'crying_baby', 'baby_cry', 'baby crying', 'infant cry',
        '아기 울음', '아이 울음', '영아', '울음소리'
    ]),
    ('doorbell_or_bell', [
        'doorbell', 'door_bell', 'bell', 'door_knock', 'knock', 'church_bells',
        '초인종', '벨소리', '노크', '문 두드림', '방문'
    ]),
]

LABEL2ID = {label: i for i, label in enumerate(TARGET_LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

def normalize_text(x):
    x = str(x).strip().lower()
    x = x.replace('-', '_').replace('/', '_').replace('\\', '_')
    x = re.sub(r'\s+', ' ', x)
    return x

def canonicalize_label(raw_label):
    """원본 데이터셋 라벨을 프로젝트 타깃 라벨로 변환합니다."""
    s = normalize_text(raw_label)
    for canonical, keys in KEYWORD_RULES:
        for key in keys:
            if normalize_text(key) in s:
                return canonical
    return 'other'

def pick_best_label(raw_labels):
    mapped = [canonicalize_label(x) for x in raw_labels if str(x).strip()]
    if not mapped:
        return 'other'
    for label in LABEL_PRIORITY:
        if label in mapped:
            return label
    return 'other'

def maybe_download_esc50(root=DATA_ROOT):
    target = root / 'ESC-50-master'
    if target.exists():
        return target
    url = 'https://github.com/karolpiczak/ESC-50/archive/master.zip'
    zip_path = root / 'ESC-50-master.zip'
    print(f'[ESC-50] downloading: {url}')
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(root)
    zip_path.unlink(missing_ok=True)
    return target

def maybe_download_urbansound8k(root=DATA_ROOT):
    target = root / 'UrbanSound8K'
    if target.exists():
        return target
    if not USE_KAGGLE_URBANSOUND8K:
        print('[UrbanSound8K] USE_KAGGLE_URBANSOUND8K=False라서 자동 다운로드를 건너뜁니다.')
        return target
    kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
    if not kaggle_json.exists():
        print('[UrbanSound8K] Kaggle 토큰이 없습니다. Colab에 kaggle.json을 업로드한 뒤 ~/.kaggle/kaggle.json에 배치하세요.')
        return target
    target.mkdir(parents=True, exist_ok=True)
    print('[UrbanSound8K] downloading from Kaggle: chrisfilo/urbansound8k')
    subprocess.run(['kaggle', 'datasets', 'download', '-d', 'chrisfilo/urbansound8k', '-p', str(target)], check=True)
    for zp in target.glob('*.zip'):
        with zipfile.ZipFile(zp, 'r') as zf:
            zf.extractall(target)
        zp.unlink(missing_ok=True)
    return target

def load_esc50(root):
    csv_candidates = list(Path(root).rglob('esc50.csv'))
    if not csv_candidates:
        return []
    meta_path = csv_candidates[0]
    base = meta_path.parent.parent
    audio_dir = base / 'audio'
    meta = pd.read_csv(meta_path)
    samples = []
    for _, row in meta.iterrows():
        raw = str(row.get('category', ''))
        label = canonicalize_label(raw)
        path = audio_dir / str(row['filename'])
        if path.exists():
            samples.append({'path': str(path), 'label': label, 'source': 'ESC-50', 'raw_label': raw})
    return samples

def load_urbansound8k(root):
    root = Path(root)
    csv_candidates = list(root.rglob('UrbanSound8K.csv'))
    if not csv_candidates:
        return []
    meta_path = csv_candidates[0]
    # 보통 .../UrbanSound8K/metadata/UrbanSound8K.csv 구조입니다.
    base = meta_path.parent.parent if meta_path.parent.name.lower() == 'metadata' else meta_path.parent
    meta = pd.read_csv(meta_path)
    samples = []
    for _, row in meta.iterrows():
        raw = str(row.get('class', row.get('classID', '')))
        label = canonicalize_label(raw)
        fname = str(row['slice_file_name'])
        fold = f"fold{int(row['fold'])}" if 'fold' in row else ''
        candidates = [base / 'audio' / fold / fname, base / fold / fname]
        path = next((p for p in candidates if p.exists()), None)
        if path is None:
            matches = list(base.rglob(fname))
            path = matches[0] if matches else None
        if path and path.exists():
            samples.append({'path': str(path), 'label': label, 'source': 'UrbanSound8K', 'raw_label': raw})
    return samples

def load_fsd50k(root):
    root = Path(root)
    if not root.exists():
        return []
    csvs = []
    for name in ['dev.csv', 'eval.csv']:
        csvs.extend(root.rglob(name))
    samples = []
    for csv_path in csvs:
        try:
            meta = pd.read_csv(csv_path)
        except Exception as e:
            print(f'[FSD50K] CSV read fail: {csv_path} / {e}')
            continue
        split = 'dev' if 'dev' in csv_path.name.lower() else 'eval'
        audio_dirs = list(root.rglob(f'FSD50K.{split}_audio')) + list(root.rglob(f'{split}_audio'))
        audio_dir = audio_dirs[0] if audio_dirs else root
        fname_col = 'fname' if 'fname' in meta.columns else meta.columns[0]
        labels_col = 'labels' if 'labels' in meta.columns else None
        if labels_col is None:
            continue
        for _, row in meta.iterrows():
            fname = str(row[fname_col])
            raw_labels = str(row[labels_col]).split(',')
            label = pick_best_label(raw_labels)
            candidates = [audio_dir / f'{fname}.wav', audio_dir / fname]
            path = next((p for p in candidates if p.exists()), None)
            if path is None:
                matches = list(root.rglob(f'{fname}.wav'))
                path = matches[0] if matches else None
            if path and path.exists():
                samples.append({'path': str(path), 'label': label, 'source': 'FSD50K', 'raw_label': ','.join(raw_labels)})
    return samples

def load_audiofolder_by_parent(root, source_name):
    """root 아래 wav/flac/mp3 파일을 찾고, 상위 폴더명을 라벨로 사용합니다. AudioSet을 직접 wav로 정리한 경우 유용합니다."""
    root = Path(root)
    if not root.exists():
        return []
    exts = ['*.wav', '*.flac', '*.mp3', '*.ogg', '*.m4a']
    files = []
    for ext in exts:
        files.extend(root.rglob(ext))
    samples = []
    for path in files:
        raw = path.parent.name
        label = canonicalize_label(raw)
        samples.append({'path': str(path), 'label': label, 'source': source_name, 'raw_label': raw})
    return samples

def flatten_values(obj):
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield str(k)
            yield from flatten_values(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from flatten_values(v)
    else:
        yield str(obj)

def load_aihub_noise(root):
    """AI Hub JSON 구조가 세부 버전마다 달라서, JSON 전체 텍스트에서 소음 라벨 키워드를 탐색합니다."""
    root = Path(root)
    if not root.exists():
        return []
    samples = []
    json_files = list(root.rglob('*.json'))
    used_audio = set()
    for jp in tqdm(json_files, desc='AI Hub JSON scan'):
        try:
            data = json.loads(jp.read_text(encoding='utf-8'))
        except Exception:
            try:
                data = json.loads(jp.read_text(encoding='cp949'))
            except Exception:
                continue
        raw_text = ' '.join(flatten_values(data))
        label = canonicalize_label(raw_text)
        # 같은 stem 또는 같은 폴더의 오디오 파일을 찾습니다.
        candidates = []
        for ext in ['.wav', '.flac', '.mp3', '.m4a']:
            candidates.append(jp.with_suffix(ext))
        if not any(p.exists() for p in candidates):
            candidates.extend(list(jp.parent.glob('*.wav')))
            candidates.extend(list(jp.parent.glob('*.flac')))
        path = next((p for p in candidates if p.exists() and str(p) not in used_audio), None)
        if path is not None:
            used_audio.add(str(path))
            samples.append({'path': str(path), 'label': label, 'source': 'AIHub_noise', 'raw_label': raw_text[:160]})
    # JSON이 없다면 폴더명 기반으로라도 로딩합니다.
    if not samples:
        samples = load_audiofolder_by_parent(root, 'AIHub_noise')
    return samples

def limit_samples(df):
    pieces = []
    for label, g in df.groupby('label'):
        cap = MAX_OTHER_SAMPLES if label == 'other' else MAX_SAMPLES_PER_CLASS
        if cap is not None and len(g) > cap:
            g = g.sample(cap, random_state=SEED)
        pieces.append(g)
    return pd.concat(pieces, ignore_index=True) if pieces else df

def safe_train_test_split(df, test_size, stratify_col='label'):
    counts = df[stratify_col].value_counts()
    stratify = df[stratify_col] if len(counts) > 1 and counts.min() >= 2 else None
    return train_test_split(df, test_size=test_size, random_state=SEED, stratify=stratify)

def build_metadata():
    all_samples = []

    if 'esc50' in DATASETS_TO_USE:
        esc_root = maybe_download_esc50(DATA_ROOT)
        s = load_esc50(esc_root)
        print(f'[ESC-50] {len(s)} samples')
        all_samples.extend(s)

    if 'urbansound8k' in DATASETS_TO_USE:
        urban_root = maybe_download_urbansound8k(DATA_ROOT)
        s = load_urbansound8k(urban_root)
        print(f'[UrbanSound8K] {len(s)} samples')
        all_samples.extend(s)

    if 'fsd50k' in DATASETS_TO_USE:
        s = load_fsd50k(FSD50K_ROOT)
        print(f'[FSD50K] {len(s)} samples from {FSD50K_ROOT}')
        all_samples.extend(s)

    if 'audioset' in DATASETS_TO_USE:
        # AudioSet은 공식적으로 YouTube segment/ontology 중심이라, wav로 준비한 폴더를 읽는 형태로 제공합니다.
        s = load_audiofolder_by_parent(AUDIOSET_ROOT, 'AudioSet_prepared')
        print(f'[AudioSet prepared audiofolder] {len(s)} samples from {AUDIOSET_ROOT}')
        all_samples.extend(s)

    if 'aihub_noise' in DATASETS_TO_USE:
        s = load_aihub_noise(AIHUB_NOISE_ROOT)
        print(f'[AI Hub noise] {len(s)} samples from {AIHUB_NOISE_ROOT}')
        all_samples.extend(s)

    df = pd.DataFrame(all_samples)
    if df.empty:
        raise RuntimeError('로드된 오디오 샘플이 없습니다. ESC-50 다운로드 상태 또는 Drive/Kaggle 경로를 확인하세요.')

    df = df.drop_duplicates('path').reset_index(drop=True)
    df = df[df['label'].isin(TARGET_LABELS)].reset_index(drop=True)
    df = limit_samples(df)

    counts = df['label'].value_counts()
    keep = counts[counts >= MIN_SAMPLES_PER_CLASS].index.tolist()
    dropped = sorted(set(df['label']) - set(keep))
    if dropped:
        print('[WARN] 샘플 수가 너무 적어 제외되는 클래스:', dropped)
        df = df[df['label'].isin(keep)].reset_index(drop=True)

    active_labels = [x for x in TARGET_LABELS if x in set(df['label'])]
    label2id = {label: i for i, label in enumerate(active_labels)}
    id2label = {i: label for label, i in label2id.items()}
    df['label_id'] = df['label'].map(label2id).astype(int)

    train_val_df, test_df = safe_train_test_split(df, test_size=0.15)
    train_df, val_df = safe_train_test_split(train_val_df, test_size=0.1765)  # 약 70/15/15

    print('\n[전체 라벨 분포]')
    print(df.groupby(['label', 'source']).size().unstack(fill_value=0))
    print('\n[split sizes]', {'train': len(train_df), 'val': len(val_df), 'test': len(test_df)})
    print('[active labels]', active_labels)
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True), active_labels, label2id, id2label

def load_audio_np(path, sr=16000, clip_seconds=5.0, random_crop=False):
    """오디오를 mono float32 [-1, 1] 근사 범위로 읽고 고정 길이로 crop/pad합니다."""
    y, _ = librosa.load(path, sr=sr, mono=True)
    if not np.isfinite(y).all():
        y = np.nan_to_num(y)
    target_len = int(sr * clip_seconds)
    if len(y) == 0:
        y = np.zeros(target_len, dtype=np.float32)
    if len(y) > target_len:
        if random_crop:
            start = random.randint(0, len(y) - target_len)
        else:
            start = max(0, (len(y) - target_len) // 2)
        y = y[start:start + target_len]
    elif len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    y = y.astype(np.float32)
    peak = np.max(np.abs(y)) if len(y) else 0
    if peak > 1.0:
        y = y / peak
    return y

def augment_waveform_np(y, sr):
    y = y.copy()
    if random.random() < 0.50:
        gain_db = random.uniform(-6.0, 6.0)
        y *= 10 ** (gain_db / 20.0)
    if random.random() < 0.35:
        shift = random.randint(-sr // 5, sr // 5)
        y = np.roll(y, shift)
    if random.random() < 0.35:
        noise_scale = random.uniform(0.0005, 0.006)
        y = y + np.random.randn(len(y)).astype(np.float32) * noise_scale
    return np.clip(y, -1.0, 1.0).astype(np.float32)

def make_class_weights(train_df, active_labels):
    labels = np.array([active_labels[i] for i in train_df['label_id'].values])
    classes = np.array(active_labels)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=labels)
    return {i: float(w) for i, w in enumerate(weights)}

def save_labels_json(path, id2label):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps({int(k): v for k, v in id2label.items()}, ensure_ascii=False, indent=2), encoding='utf-8')
    print('saved:', path)

## 데이터셋 로딩

In [7]:
train_df, val_df, test_df, active_labels, label2id, id2label = build_metadata()
class_weights = make_class_weights(train_df, active_labels)
print('class_weights:', class_weights)

[ESC-50] 2000 samples
[UrbanSound8K] USE_KAGGLE_URBANSOUND8K=False라서 자동 다운로드를 건너뜁니다.
[UrbanSound8K] 0 samples
[FSD50K] 0 samples from /content/drive/MyDrive/datasets/FSD50K
[AudioSet prepared audiofolder] 0 samples from /content/drive/MyDrive/datasets/AudioSet_prepared
[AI Hub noise] 0 samples from /content/drive/MyDrive/datasets/AIHub_noise_speech

[전체 라벨 분포]
source                   ESC-50
label                          
baby_cry                     40
car_horn                     40
construction_or_machine     120
doorbell_or_bell             80
explosion_or_gunshot         40
fire_alarm                   40
glass_breaking               40
other                       600
siren                        40

[split sizes] {'train': 727, 'val': 157, 'test': 156}
[active labels] ['car_horn', 'siren', 'glass_breaking', 'explosion_or_gunshot', 'construction_or_machine', 'fire_alarm', 'baby_cry', 'doorbell_or_bell', 'other']
class_weights: {0: 2.884920634920635, 1: 2.884920634920635, 2: 2.884

## Hugging Face Dataset 변환

In [8]:
import torch
from torch import nn
from datasets import Dataset
from transformers import ASTFeatureExtractor, ASTForAudioClassification, TrainingArguments, Trainer
import evaluate

SAMPLE_RATE = 16000
CLIP_SECONDS = 10.0
BATCH_SIZE = 4
EPOCHS = 4
LR = 2e-5
FREEZE_BACKBONE = True
MODEL_NAME = 'MIT/ast-finetuned-audioset-10-10-0.4593'

feature_extractor = ASTFeatureExtractor.from_pretrained(MODEL_NAME)

def df_to_hfds(df):
    return Dataset.from_pandas(df[['path', 'label_id', 'label', 'source']].reset_index(drop=True), preserve_index=False)

train_ds = df_to_hfds(train_df)
val_ds = df_to_hfds(val_df)
test_ds = df_to_hfds(test_df)

def preprocess_batch(batch, train=False):
    arrays = [load_audio_np(p, sr=SAMPLE_RATE, clip_seconds=CLIP_SECONDS, random_crop=train) for p in batch['path']]
    features = feature_extractor(arrays, sampling_rate=SAMPLE_RATE, return_tensors='np')
    batch['input_values'] = features['input_values']
    batch['labels'] = batch['label_id']
    return batch

train_ds = train_ds.map(lambda b: preprocess_batch(b, train=True), batched=True, batch_size=16)
val_ds = val_ds.map(lambda b: preprocess_batch(b, train=False), batched=True, batch_size=16)
test_ds = test_ds.map(lambda b: preprocess_batch(b, train=False), batched=True, batch_size=16)

cols = ['input_values', 'labels']
train_ds.set_format(type='torch', columns=cols)
val_ds.set_format(type='torch', columns=cols)
test_ds.set_format(type='torch', columns=cols)

model = ASTForAudioClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(active_labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

if FREEZE_BACKBONE:
    for name, p in model.named_parameters():
        if not name.startswith('classifier'):
            p.requires_grad = False

class_weight_tensor = torch.tensor([class_weights.get(i, 1.0) for i in range(len(active_labels))], dtype=torch.float32)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss = nn.CrossEntropyLoss(weight=class_weight_tensor.to(logits.device))(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro', zero_division=0),
    }

try:
    training_args = TrainingArguments(
        output_dir='/content/ast_danger_sound',
        evaluation_strategy='epoch',
        save_strategy='epoch',
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=1e-4,
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        logging_steps=25,
        disable_tqdm=True,
        report_to=[],
        fp16=torch.cuda.is_available(),
    )
except TypeError:
    training_args = TrainingArguments(
        output_dir='/content/ast_danger_sound',
        eval_strategy='epoch',
        save_strategy='epoch',
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=1e-4,
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        logging_steps=25,
        disable_tqdm=True,
        report_to=[],
        fp16=torch.cuda.is_available(),
    )

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=feature_extractor,
    compute_metrics=compute_metrics,
)

Map:   0%|          | 0/727 [00:00<?, ? examples/s]

Map:   0%|          | 0/157 [00:00<?, ? examples/s]

Map:   0%|          | 0/156 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                       
------------------------+----------+---------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([9])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([9, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


## 학습 / 평가

In [9]:
trainer.train()
print(trainer.evaluate(test_ds))

pred = trainer.predict(test_ds)
y_true = pred.label_ids
y_pred = pred.predictions.argmax(axis=-1)
print(classification_report(y_true, y_pred, target_names=active_labels, zero_division=0))
print('confusion_matrix:\n', confusion_matrix(y_true, y_pred))

{'loss': '2.509', 'grad_norm': '26.79', 'learning_rate': '1.934e-05', 'epoch': '0.1374'}
{'loss': '2.39', 'grad_norm': '21.13', 'learning_rate': '1.865e-05', 'epoch': '0.2747'}
{'loss': '2.39', 'grad_norm': '21.13', 'learning_rate': '1.865e-05', 'epoch': '0.2747'}
{'loss': '2.386', 'grad_norm': '21.76', 'learning_rate': '1.797e-05', 'epoch': '0.4121'}
{'loss': '2.386', 'grad_norm': '21.76', 'learning_rate': '1.797e-05', 'epoch': '0.4121'}
{'loss': '2.401', 'grad_norm': '21.62', 'learning_rate': '1.728e-05', 'epoch': '0.5495'}
{'loss': '2.401', 'grad_norm': '21.62', 'learning_rate': '1.728e-05', 'epoch': '0.5495'}
{'loss': '2.37', 'grad_norm': '20.94', 'learning_rate': '1.659e-05', 'epoch': '0.6868'}
{'loss': '2.37', 'grad_norm': '20.94', 'learning_rate': '1.659e-05', 'epoch': '0.6868'}
{'loss': '2.156', 'grad_norm': '19.75', 'learning_rate': '1.591e-05', 'epoch': '0.8242'}
{'loss': '2.156', 'grad_norm': '19.75', 'learning_rate': '1.591e-05', 'epoch': '0.8242'}
{'loss': '2.109', 'grad_n

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '2.189', 'grad_norm': '19.68', 'learning_rate': '1.453e-05', 'epoch': '1.099'}
{'loss': '2.051', 'grad_norm': '18.42', 'learning_rate': '1.385e-05', 'epoch': '1.236'}
{'loss': '2.051', 'grad_norm': '18.42', 'learning_rate': '1.385e-05', 'epoch': '1.236'}
{'loss': '2.086', 'grad_norm': '23.02', 'learning_rate': '1.316e-05', 'epoch': '1.374'}
{'loss': '2.086', 'grad_norm': '23.02', 'learning_rate': '1.316e-05', 'epoch': '1.374'}
{'loss': '2.036', 'grad_norm': '19.91', 'learning_rate': '1.247e-05', 'epoch': '1.511'}
{'loss': '2.036', 'grad_norm': '19.91', 'learning_rate': '1.247e-05', 'epoch': '1.511'}
{'loss': '2.121', 'grad_norm': '24.84', 'learning_rate': '1.179e-05', 'epoch': '1.648'}
{'loss': '2.121', 'grad_norm': '24.84', 'learning_rate': '1.179e-05', 'epoch': '1.648'}
{'loss': '2.123', 'grad_norm': '25.3', 'learning_rate': '1.11e-05', 'epoch': '1.786'}
{'loss': '2.123', 'grad_norm': '25.3', 'learning_rate': '1.11e-05', 'epoch': '1.786'}
{'loss': '2.033', 'grad_norm': '15.9

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '1.933', 'grad_norm': '16.04', 'learning_rate': '9.725e-06', 'epoch': '2.06'}
{'loss': '2.014', 'grad_norm': '17.68', 'learning_rate': '9.038e-06', 'epoch': '2.198'}
{'loss': '2.014', 'grad_norm': '17.68', 'learning_rate': '9.038e-06', 'epoch': '2.198'}
{'loss': '1.937', 'grad_norm': '17.56', 'learning_rate': '8.352e-06', 'epoch': '2.335'}
{'loss': '1.937', 'grad_norm': '17.56', 'learning_rate': '8.352e-06', 'epoch': '2.335'}
{'loss': '1.871', 'grad_norm': '17.27', 'learning_rate': '7.665e-06', 'epoch': '2.473'}
{'loss': '1.871', 'grad_norm': '17.27', 'learning_rate': '7.665e-06', 'epoch': '2.473'}
{'loss': '1.806', 'grad_norm': '19.24', 'learning_rate': '6.978e-06', 'epoch': '2.61'}
{'loss': '1.806', 'grad_norm': '19.24', 'learning_rate': '6.978e-06', 'epoch': '2.61'}
{'loss': '1.848', 'grad_norm': '25.67', 'learning_rate': '6.291e-06', 'epoch': '2.747'}
{'loss': '1.848', 'grad_norm': '25.67', 'learning_rate': '6.291e-06', 'epoch': '2.747'}
{'loss': '1.905', 'grad_norm': '20.

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '1.835', 'grad_norm': '20.41', 'learning_rate': '4.918e-06', 'epoch': '3.022'}
{'loss': '1.842', 'grad_norm': '25.18', 'learning_rate': '4.231e-06', 'epoch': '3.159'}
{'loss': '1.842', 'grad_norm': '25.18', 'learning_rate': '4.231e-06', 'epoch': '3.159'}
{'loss': '1.815', 'grad_norm': '16.98', 'learning_rate': '3.544e-06', 'epoch': '3.297'}
{'loss': '1.815', 'grad_norm': '16.98', 'learning_rate': '3.544e-06', 'epoch': '3.297'}
{'loss': '1.806', 'grad_norm': '19.36', 'learning_rate': '2.857e-06', 'epoch': '3.434'}
{'loss': '1.806', 'grad_norm': '19.36', 'learning_rate': '2.857e-06', 'epoch': '3.434'}
{'loss': '1.78', 'grad_norm': '16.72', 'learning_rate': '2.17e-06', 'epoch': '3.571'}
{'loss': '1.78', 'grad_norm': '16.72', 'learning_rate': '2.17e-06', 'epoch': '3.571'}
{'loss': '1.822', 'grad_norm': '18.25', 'learning_rate': '1.484e-06', 'epoch': '3.709'}
{'loss': '1.822', 'grad_norm': '18.25', 'learning_rate': '1.484e-06', 'epoch': '3.709'}
{'loss': '1.732', 'grad_norm': '14.9

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '119.8', 'train_samples_per_second': '24.28', 'train_steps_per_second': '6.078', 'train_loss': '2.027', 'epoch': '4'}
{'eval_loss': '1.754', 'eval_accuracy': '0.4936', 'eval_macro_f1': '0.2583', 'eval_runtime': '4.735', 'eval_samples_per_second': '32.94', 'eval_steps_per_second': '8.236', 'epoch': '4'}
{'eval_loss': 1.7540830373764038, 'eval_accuracy': 0.4935897435897436, 'eval_macro_f1': 0.25828139551543805, 'eval_runtime': 4.7353, 'eval_samples_per_second': 32.944, 'eval_steps_per_second': 8.236, 'epoch': 4.0}
{'eval_loss': '1.754', 'eval_accuracy': '0.4936', 'eval_macro_f1': '0.2583', 'eval_runtime': '4.735', 'eval_samples_per_second': '32.94', 'eval_steps_per_second': '8.236', 'epoch': '4'}
{'eval_loss': 1.7540830373764038, 'eval_accuracy': 0.4935897435897436, 'eval_macro_f1': 0.25828139551543805, 'eval_runtime': 4.7353, 'eval_samples_per_second': 32.944, 'eval_steps_per_second': 8.236, 'epoch': 4.0}
                         precision    recall  f1-score   support

## 파일 하나 예측 / 저장

In [10]:
def predict_file_ast(audio_path, top_k=5):
    arr = load_audio_np(audio_path, sr=SAMPLE_RATE, clip_seconds=CLIP_SECONDS, random_crop=False)
    inputs = feature_extractor([arr], sampling_rate=SAMPLE_RATE, return_tensors='pt')
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
    order = probs.argsort()[::-1]
    return [(active_labels[i], float(probs[i])) for i in order[:top_k]]

sample_path = test_df.iloc[0]['path']
print(sample_path, test_df.iloc[0]['label'])
print(predict_file_ast(sample_path))

trainer.save_model('/content/ast_danger_sound_best')
feature_extractor.save_pretrained('/content/ast_danger_sound_best')
save_labels_json('/content/ast_danger_sound_best/labels.json', id2label)
print('saved /content/ast_danger_sound_best')

/content/sound_data/ESC-50-master/audio/4-204115-A-39.wav glass_breaking
[('baby_cry', 0.16747534275054932), ('glass_breaking', 0.16161088645458221), ('other', 0.1535714864730835), ('construction_or_machine', 0.13402922451496124), ('car_horn', 0.10761993378400803)]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved: /content/ast_danger_sound_best/labels.json
saved /content/ast_danger_sound_best
